***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import geopandas as gpd


from time import perf_counter as perf
import pyodbc
import urllib
import sqlalchemy as sqla

# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths



if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_geo = os.path.join(path_users, 'Documents', 'Geospatial Data')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'RTIS Data')
    path_main = os.path.join(path_sp, 'Data')
    path_congestion = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Congestion')



path_config0 = os.path.join(path_git, 'config')
path_code    = os.path.join(path_git, 'Data', 'RTIS')
path_config  = os.path.join(path_code, 'config')
path_sql     = os.path.join(path_git, 'Data', 'RTIS', 'SQL Scripts')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Congestion_2

***

In [ ]:
# Keep track of time amounted while SQL query is running through all years
start_time = time.time()

# Set up parameters for SQL queries
db = 'NPMRDS'
year_start = 2017
year_end   = 2023
years_to_import = range(year_start, year_end+1)
list_df_sql = []

# Loop through years to calculate % road miles congested by year
for year in years_to_import:

    print('')
    print(f'Running congestion calculator on year {year}...')
    print('')

    with open(os.path.join(path_sql, f'Congestion_2_{year}_pct_rd_miles_congested.sql'), 'r') as query:
        query_string = query.read()
    
    df_sql = sqlqry_to_df(query_string, db)
    df_sql['Year'] = year
    list_df_sql.append(df_sql)
    print('')

# Calculate time
print('')
print('Finished!!')
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")
print('')


# Concatenate all years together
df_congestion2 = pd.concat(list_df_sql)
display(df_congestion2)

In [ ]:
df_congestion2 = pd.concat(list_df_sql)

df_congestion2['MPO'] = 'SACOG'
df_congestion2 = df_congestion2.set_index(['MPO', 'Year']).reset_index()
df_congestion2 = df_congestion2.sort_values('Year', ascending = False)
df_congestion2['pct_dirmi_congested'] = df_congestion2['pct_dirmi_congested']*100
df_congestion2

In [ ]:
# Congestion_1 Exports to SP
indicator_name = 'Congestion_2'
sample_type = 'RTIS'
year_start = 2017
year_end = 2023
geography = 'MPO'
df_about = write_about(sample_type, indicator_name, geography, year_start, year_end, path_config0)
path_out = os.path.join(path_congestion, indicator_name)
workbook_name = f"{indicator_name} SACOG RTIS.xlsx"
with pd.ExcelWriter(os.path.join(path_out, workbook_name), engine='xlsxwriter') as writer:
    df_about      .to_excel(writer, index = False, sheet_name = 'About', header=False)
    df_congestion2.to_excel(writer, index = False, sheet_name = 'SACOG'              )

In [ ]:
df_congestion2.columns = [col.lower() for col in df_congestion2.columns]
df_congestion2.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_congestion2.columns]
df_congestion2.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_congestion2.columns]

# df_congestion2.to_csv(os.path.join(path_agol, 'Congestion_2_SACOG_RTIS.csv'), index = False)
